# P11 — Baseline Ekstraktif Sederhana dengan TF-IDF

Notebook ini membangun baseline ringkasan ekstraktif sebagai pembanding untuk model abstraktif T5 pada P12.

**Input:** satu file `*_token_chunks.json` dari P09.  
**Output:** ringkasan baseline dan skor TF-IDF per kalimat.

Catatan: baseline ini memilih kalimat dari teks sumber, bukan menghasilkan parafrasa baru. Jalankan pada dokumen pengembangan terlebih dahulu. Jangan memakai dokumen test untuk tuning parameter.


## 1. Instalasi dan impor

In [15]:
!pip -q install scikit-learn

from pathlib import Path
import json
import sys
import re
import pandas as pd
from google.colab import files

# Sesuaikan jika nama folder repository di Drive berbeda.
REPO_ROOT = Path('/content/drive/MyDrive/NLPKelompok3A-main')
SRC_DIR = REPO_ROOT / 'src'

print('Repository ditemukan:', REPO_ROOT.exists())
print('Modul P11 ditemukan:', (SRC_DIR / 'baseline_extractive.py').exists())

if SRC_DIR.exists():
    print("Isi folder src:")
    print([f.name for f in SRC_DIR.iterdir()])

Repository ditemukan: True
Modul P11 ditemukan: True
Isi folder src:
['baseline_extractive.py', '__pycache__']


In [16]:
import sys
import importlib
import importlib.util
from pathlib import Path

REPO_ROOT = Path("/content/drive/MyDrive/NLPKelompok3A-main")
SRC_DIR = REPO_ROOT / "src"
MODULE_FILE = SRC_DIR / "baseline_extractive.py"

print("File ditemukan:", MODULE_FILE.exists())
print("Lokasi file:", MODULE_FILE.resolve())

# Masukkan folder src ke Python path
sys.path.insert(0, str(SRC_DIR.resolve()))

# Perbarui cache pencarian modul
importlib.invalidate_caches()

# Import modul secara langsung
spec = importlib.util.spec_from_file_location(
    "baseline_extractive",
    MODULE_FILE
)

module = importlib.util.module_from_spec(spec)
sys.modules["baseline_extractive"] = module
spec.loader.exec_module(module)

load_p09_records = module.load_p09_records
build_extractive_summary = module.build_extractive_summary

print("Import P11 berhasil!")

File ditemukan: True
Lokasi file: /content/drive/MyDrive/NLPKelompok3A-main/src/baseline_extractive.py
Import P11 berhasil!


## 2. Hubungkan Google Drive dan impor modul P11

In [17]:
from google.colab import drive
drive.mount('/content/drive')

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from baseline_extractive import load_p09_records, build_extractive_summary


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Unggah satu hasil JSON dari P09

In [18]:
print('Pilih satu file *_token_chunks.json dari hasil notebook P09.')
uploaded = files.upload()

if not uploaded:
    raise RuntimeError('Tidak ada file yang diunggah.')
if len(uploaded) != 1:
    raise ValueError('Unggah satu file P09 saja untuk setiap eksperimen.')

input_filename, raw_bytes = next(iter(uploaded.items()))
try:
    p09_payload = json.loads(raw_bytes.decode('utf-8'))
except Exception as exc:
    raise ValueError(f'JSON tidak valid: {exc}') from exc

records = load_p09_records(p09_payload)
print('File:', input_filename)
print('Jumlah chunk:', len(p09_payload.get('chunks', [])))
print('Jumlah unit kalimat/fragmen unik:', len(records))
print('Pratinjau:')
for row in records[:5]:
    print(f"{row.get('fragment_id')} | hal. {row.get('page')} | {row['text'][:220]}")


Pilih satu file *_token_chunks.json dari hasil notebook P09.


Saving Alternifa_-_Memahami_Informasi_Artikel_token_chunks.json to Alternifa_-_Memahami_Informasi_Artikel_token_chunks.json
File: Alternifa_-_Memahami_Informasi_Artikel_token_chunks.json
Jumlah chunk: 14
Jumlah unit kalimat/fragmen unik: 580
Pratinjau:
S00001 | hal. 1 | Pulang Mudik: Selebrasi Agama dan Budaya
S00002 | hal. 1 | Acara pulang mudik Lebaran selalu disambut dengan antusias oleh masyarakat layaknya sebuah selebrasi kolosal.
S00003 | hal. 1 | Pulang mudik bukan sekadar melepas rindu kampung halaman, melainkan juga dihayati sebagai rangkaian rasa syukur dengan berakhirnya ibadah puasa selama sebulan.
S00004 | hal. 1 | Ada keyakinan bagi mereka yang berpuasa bahwa ampunan Tuhan yang dijanjikan dalam bulan Ramadhan akan terganjal jika antara anak dan orangtua dan antara sesama teman serta keluarga belum saling memaafkan.
S00005 | hal. 1 | Makanya acara pulang mudik juga punya makna spiritual yang kental, di samping melepas rindu dan napak tilas dengan kampung halaman.


## 4. Tentukan batas panjang ringkasan

Angka di bawah adalah batas maksimum kata untuk eksperimen baseline, bukan jaminan jumlah kata yang harus selalu tercapai. Kamu dapat menjalankan ulang dengan batas berbeda untuk membandingkan tiga tingkat panjang.


In [19]:
SUMMARY_BUDGETS = {
    'pendek': 80,
    'sedang': 150,
    'panjang': 250,
}

print(SUMMARY_BUDGETS)


{'pendek': 80, 'sedang': 150, 'panjang': 250}


## 5. Jalankan baseline TF-IDF untuk tiga tingkat panjang

In [20]:
baseline_results = {}

for level, word_budget in SUMMARY_BUDGETS.items():
    baseline_results[level] = build_extractive_summary(
        records,
        max_words=word_budget
    )
    result = baseline_results[level]
    print(f"\\n===== {level.upper()} | batas {word_budget} kata =====")
    print(result['summary'])
    print(
        f"\nKata aktual: {result['actual_words']} | "
        f"Kalimat terpilih: {result['selected_sentence_count']}"
    )


\n===== PENDEK | batas 80 kata =====
www.alternatifa.com Jemaah www.alternatifa.com www.alternatifa.com www.alternatifa.com www.alternatifa.com www.alternatifa.com www.alternatifa.com www.alternatifa.com www.alternatifa.com www.alternatifa.com www.alternatifa.com Sumber: Komunikasi Pemerintah dan Ketidakpekaan terhadap Krisis di Masyarakat www.alternatifa.com www.alternatifa.com www.alternatifa.com www.alternatifa.com www.alternatifa.com www.alternatifa.com www.alternatifa.com www.alternatifa.com www.alternatifa.com www.alternatifa.com Meski masih mewajibkan ”pengunduran diri” atau ”pensiun dini” bagi kekaryaan militer (Pasal 47 Ayat 1), UU TNI lama yang mereka buat bersama wakil militer di parlemen itu membuka celah kembalinya fungsi kekaryaan militer itu pada sepuluh jabatan sipil (Pasal 47 Ayat 2). www.alternatifa.com www.alternatifa.com Sumber: Kebebasan Berbicara www.alternatifa.com www.alternatifa.com

Kata aktual: 80 | Kalimat terpilih: 29
\n===== SEDANG | batas 150 kata =====
w

/usr/local/lib/python3.13/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['an', 'ang'] not in stop_words.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['an', 'ang'] not in stop_words.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['an', 'ang'] not in stop_words.
  warnings.warn(


## 6. Periksa kalimat terpilih dan skor TF-IDF

In [21]:
selected_rows = []
for level, result in baseline_results.items():
    for item in result['source_sentences']:
        selected_rows.append({
            'tingkat': level,
            'fragment_id': item['fragment_id'],
            'halaman': item.get('page'),
            'skor_tfidf': item['tfidf_score'],
            'teks': item['text'],
        })

selected_df = pd.DataFrame(selected_rows)
display(selected_df)


,tingkat,fragment_id,halaman,skor_tfidf,teks
0,pendek,S00022,1,0.501710,www.alternatifa.com
1,pendek,S00048,2,0.020068,Jemaah
2,pendek,S00049,2,0.501710,www.alternatifa.com
3,pendek,S00072,3,0.501710,www.alternatifa.com
4,pendek,S00092,4,0.501710,www.alternatifa.com
...,...,...,...,...,...
87,panjang,S00518,22,0.127428,Sepuluh jabatan sipil itu ialah kantor yang me...
88,panjang,S00522,22,0.501710,www.alternatifa.com
89,panjang,S00535,23,0.501710,www.alternatifa.com
90,panjang,S00560,24,0.501710,www.alternatifa.com


## 7. Simpan hasil eksperimen

In [22]:
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'baseline_summaries'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

source_stem = Path(input_filename).stem.replace('_token_chunks', '')
output_payload = {
    'method': 'TF-IDF extractive baseline',
    'source_file': p09_payload.get('pipeline_metadata', {}).get('source_file', input_filename),
    'input_json': input_filename,
    'input_chunk_count': len(p09_payload.get('chunks', [])),
    'unique_source_units': len(records),
    'summary_budgets': SUMMARY_BUDGETS,
    'results': baseline_results,
}

output_json = OUTPUT_DIR / f'{source_stem}_baseline_tfidf.json'
output_json.write_text(
    json.dumps(output_payload, ensure_ascii=False, indent=2),
    encoding='utf-8'
)

# Simpan juga versi teks agar mudah dibaca.
output_txt = OUTPUT_DIR / f'{source_stem}_baseline_tfidf.txt'
with output_txt.open('w', encoding='utf-8') as f:
    f.write(f'Sumber: {input_filename}\\nMetode: TF-IDF ekstraktif\\n')
    for level, result in baseline_results.items():
        f.write(f"\\n===== {level.upper()} =====\\n")
        f.write(result['summary'] + '\\n')
        f.write(
            f"\\nKata aktual: {result['actual_words']} | "
            f"Kalimat terpilih: {result['selected_sentence_count']}\\n"
        )

print('JSON:', output_json)
print('TXT :', output_txt)
files.download(str(output_json))
files.download(str(output_txt))


JSON: /content/drive/MyDrive/NLPKelompok3A-main/outputs/baseline_summaries/Alternifa_-_Memahami_Informasi_Artikel_baseline_tfidf.json
TXT : /content/drive/MyDrive/NLPKelompok3A-main/outputs/baseline_summaries/Alternifa_-_Memahami_Informasi_Artikel_baseline_tfidf.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Interpretasi dan batasan

- Ringkasan dibentuk dari kalimat sumber yang mendapat skor relevansi TF-IDF lebih tinggi terhadap representasi dokumen.
- Kalimat terpilih dikembalikan ke urutan aslinya agar alur bacaan tidak mengikuti urutan skor.
- Duplikasi akibat overlap chunk P09 dikurangi menggunakan `fragment_id`.
- Baseline ekstraktif tidak melakukan parafrasa. Kalimat yang terpilih bisa kurang koheren jika konteks antarkalimat berjauhan.
- Periksa hasil secara manual, terutama jika PDF mengandung tabel, daftar, rumus, atau hasil ekstraksi yang tidak rapi.
- Simpan hasil eksperimen pada dokumen **development**. Dokumen **test** dipertahankan untuk evaluasi setelah metode diputuskan.
